# Notebook 00: Setup Verification

**Time:** ~5 minutes  
**Goal:** Verify that your Python environment is ready for the Week 4 RAG homework.

Run every cell top-to-bottom. Any ❌ means you need to install/configure something before continuing to Notebook 01.


## 1. Python version


In [1]:
import sys
v = sys.version_info
print(f'Python {v.major}.{v.minor}.{v.micro}')
assert v >= (3, 9), 'Python 3.9+ required (3.10/3.11 recommended for sentence-transformers)'
print('✓ Python version OK')


Python 3.10.10
✓ Python version OK


## 2. Core packages (always required)


In [3]:
import importlib
core = ['jupyter', 'anthropic', 'requests', 'dotenv', 'pydantic', 'numpy']
for p in core:
    try:
        importlib.import_module(p)
        print(f'  ✓ {p}')
    except ImportError:
        print(f'  ❌ {p}  -- please install the package with pip install -r ../requirements.txt')


  ✓ jupyter
  ✓ anthropic
  ✓ requests
  ✓ dotenv
  ✓ pydantic
  ✓ numpy


## 3. Week 4 packages (RAG stack)


In [4]:
rag_pkgs = {
    'fitz':                 'PyMuPDF (PDF text extraction)',
    'sentence_transformers':'sentence-transformers (embeddings + cross-encoders)',
    'faiss':                'faiss-cpu (vector index)',
    'chromadb':             'chromadb (persistent vector DB)',
    'rank_bm25':            'rank-bm25 (sparse retrieval for hybrid search)',
    'tiktoken':             'tiktoken (token counting)',
    'flashrank':            'flashrank (lightweight reranker, optional)',
    'bs4':                  'beautifulsoup4 (web loader, optional)',
}
missing = []
for mod, desc in rag_pkgs.items():
    try:
        importlib.import_module(mod)
        print(f'  ✓ {mod:25s} -- {desc}')
    except ImportError:
        print(f'  ⚠ {mod:25s} -- MISSING ({desc})')
        missing.append(mod)
if missing:
    print(f'\nInstall: pip install -r ../requirements.txt')


  ✓ fitz                      -- PyMuPDF (PDF text extraction)
  ✓ sentence_transformers     -- sentence-transformers (embeddings + cross-encoders)
  ✓ faiss                     -- faiss-cpu (vector index)
  ✓ chromadb                  -- chromadb (persistent vector DB)
  ✓ rank_bm25                 -- rank-bm25 (sparse retrieval for hybrid search)
  ✓ tiktoken                  -- tiktoken (token counting)
  ✓ flashrank                 -- flashrank (lightweight reranker, optional)
  ✓ bs4                       -- beautifulsoup4 (web loader, optional)


## 4. Environment variables

Path A (Claude) needs `ANTHROPIC_API_KEY`. Path B (Ollama) needs none.


In [9]:
import os
from dotenv import load_dotenv
load_dotenv('../.env', override=True)
for k in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'COHERE_API_KEY', 'VOYAGE_API_KEY', 'HF_TOKEN']:
    val = os.getenv(k)
    print(f'  {"✓" if val else " "} {k:22s} = {val[:8] + "..." if val else "(not set)"}')
if not os.getenv('ANTHROPIC_API_KEY'):
    print('\n⚠ ANTHROPIC_API_KEY not set. Path A will fail until you create .env (cp .env.example .env).')


  ✓ ANTHROPIC_API_KEY      = sk-ant-y...
  ✓ OPENAI_API_KEY         = sk-proj-...
    COHERE_API_KEY         = (not set)
    VOYAGE_API_KEY         = (not set)
    HF_TOKEN               = (not set)


### Note:   
The homework's default path uses local sentence-transformers embeddings + a local cross-encoder reranker, which run on CPU and cost nothing. 

The COHERE_API and VOYAGE are both optional API keys for alternative embedding/reranking backends in your RAG stack:
- VOYAGE_API_KEY — used in 
    - src/embeddings.py:99 for Voyage AI embeddings (voyage-3-large, top-tier quality)
- COHERE_API_KEY — used in:
    - src/embeddings.py:107 for Cohere embeddings (embed-v4.0, multilingual)
    - src/reranker.py:124 for Cohere Rerank 4.0 (the highest-accuracy reranker)
   
The API keys exist so students who want to can benchmark their local setup against premium paid services. 
you can get the API from the following link:

* VOYAGE API: https://dashboard.voyageai.com/organization/projects

* COHERE API: https://dashboard.cohere.com/

## 5. Ollama (optional — Path B)


In [9]:
import requests
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    if r.ok:
        models = [m['name'] for m in r.json().get('models', [])]
        print(f'✓ Ollama running. Models: {models or "(none — pull qwen3.5:27b or llama3.1:8b)"}')
    else:
        print(f'⚠ Ollama returned {r.status_code}')
except Exception as e:
    print(f'  Ollama not running ({e}). That\'s fine if you\'re using Path A.')


✓ Ollama running. Models: ['qwen3.5:27b', 'qwen3.5:9b']


## 6. Directory structure


In [8]:
from pathlib import Path
root = Path('..').resolve()
for sub in ['notebooks', 'src', 'outputs', 'docs', 'test_data']:
    p = root / sub
    print(f'  {"✓" if p.exists() else "❌"} {sub}/')
    if not p.exists():
        p.mkdir(exist_ok=True)
        print(f'    -> created')


  ✓ notebooks/
  ✓ src/
  ✓ outputs/
  ✓ docs/
  ✓ test_data/


## 7. Sample test data

We've shipped a small resume + portfolio fixture. You'll use real data of your own choice in nb02.


In [8]:
import os
for f in ['sample_resume.pdf', 'portfolio_notes.txt']:
    p = os.path.join('..', 'test_data', f)
    print(f'  {"✓" if os.path.exists(p) else "❌"} test_data/{f}')


  ✓ test_data/sample_resume.pdf
  ✓ test_data/portfolio_notes.txt


## ✅ Setup checklist

If all the boxes above are ticked:

1. **Path A users:** make sure `.env` has your Anthropic key.
2. **Path B users:** `ollama serve` running with `qwen3.5:27b` or `llama3.1:8b` pulled.
3. **All:** continue to **Notebook 01: Environment Setup**.

**Common fixes:**
- `pip install -r ../requirements.txt` for any missing packages
- `cp ../.env.example ../.env` then edit
- `ollama pull qwen3.5:27b` to fetch the local model (~17GB)
- For Apple Silicon: `pip install faiss-cpu` (the GPU version isn't on macOS)
